# Visual check: DGX 50k depth + normals

Mount Google Drive, load the copied best checkpoint and run manifest, then infer five deterministic held-out test samples. This notebook runs inference only; it does not train or copy the full dataset into Colab runtime.

Expected Drive folders: `AppleCapstoneRelighting/results/fastvit_t8_fpn_depth_normals_local50000_seed42_v1` and `AppleCapstoneRelighting/dataset/sources/SynthHuman`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE_PROJECT = Path('/content/drive/MyDrive/AppleCapstoneRelighting')
RUN_DIR = DRIVE_PROJECT / 'results/fastvit_t8_fpn_depth_normals_local50000_seed42_v1'
SOURCE_ROOT = DRIVE_PROJECT / 'dataset/sources/SynthHuman'
print('Drive project:', DRIVE_PROJECT)
print('Run results:', RUN_DIR)
print('SynthHuman source:', SOURCE_ROOT)

## Install runtime dependencies and import the same model/data code used for training

The model is initialized with `pretrained=False`: all learned weights come from the saved checkpoint, so no pretrained model download is needed.

In [ ]:
%pip -q install 'timm>=1.0,<2' 'OpenEXR>=3.3,<4' pyyaml google-api-python-client google-auth

import sys, subprocess, json, random, yaml
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
REPO_DIR = Path('/content/relight-train-model')
if not (REPO_DIR / 'src/relight_train/model.py').is_file():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/MuhammadAliffandy/relight-train-model.git', str(REPO_DIR)], check=True)
sys.path.insert(0, str(REPO_DIR / 'src'))
from relight_train.data import RawSynthHumanDataset
from relight_train.model import PortraitGeometryModel
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE, '| GPU:', torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'CPU')

## Lima sampel SynthHuman di luar manifest run

Cell berikut membaca manifest run 50k, memeriksa metadata file hanya di `chunk_0026`, lalu memilih lima ID valid yang tidak ada di manifest train/val/test run itu. Jadi perbandingan GT vs prediksi memakai sampel SynthHuman yang tidak dipakai oleh run tersebut. Pemilihan berdasarkan ID sampel; ini tidak menjamin identitas manusia yang benar-benar berbeda karena metadata identitas subjek tidak tersedia di manifest.

Yang diunduh hanya checkpoint, config, manifest, dan empat modality untuk lima sampel.


In [ ]:
# Pilih lima sampel deterministik dari satu chunk, setelah mengecualikan semua ID dalam manifest run.
CHUNK_NAME = 'chunk_0026'
LOCAL_DATA = Path('/content/relight_visual_test')
LOCAL_DATA.mkdir(parents=True, exist_ok=True)

from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
from google.auth import default
import io, re

auth.authenticate_user()
credentials, _ = default()
drive_api = build('drive', 'v3', credentials=credentials, cache_discovery=False)
FOLDER_MIME = 'application/vnd.google-apps.folder'
SHORTCUT_MIME = 'application/vnd.google-apps.shortcut'

def child_folder_id(parent_id, name):
    query = f"'{parent_id}' in parents and name = '{name}' and trashed = false"
    response = drive_api.files().list(
        q=query, pageSize=100,
        fields='files(id,name,mimeType,shortcutDetails)',
        includeItemsFromAllDrives=True, supportsAllDrives=True,
    ).execute()
    for item in response.get('files', []):
        if item['mimeType'] == FOLDER_MIME:
            return item['id']
        if item['mimeType'] == SHORTCUT_MIME and item.get('shortcutDetails', {}).get('targetMimeType') == FOLDER_MIME:
            return item['shortcutDetails']['targetId']
    raise FileNotFoundError(f'Drive folder not found: {name} under parent ID {parent_id}')

def find_named_files(parent_id, names):
    name_query = ' or '.join("name = '{}'".format(name.replace("'", "\\'")) for name in names)
    query = f"'{parent_id}' in parents and ({name_query}) and trashed = false"
    response = drive_api.files().list(
        q=query, pageSize=1000,
        fields='files(id,name,mimeType,size)',
        includeItemsFromAllDrives=True, supportsAllDrives=True,
    ).execute()
    found = {item['name']: item for item in response.get('files', [])}
    missing = sorted(set(names) - found.keys())
    if missing:
        raise FileNotFoundError(f'Missing expected Drive files: {missing}')
    return found

def list_chunk_files(parent_id):
    # Metadata only: list one 5k-sample chunk, not file contents or all Drive entries.
    found = {}
    page_token = None
    while True:
        response = drive_api.files().list(
            q=f"'{parent_id}' in parents and trashed = false",
            pageSize=1000, pageToken=page_token,
            fields='nextPageToken,files(id,name,mimeType)',
            includeItemsFromAllDrives=True, supportsAllDrives=True,
        ).execute()
        found.update({item['name']: item for item in response.get('files', [])})
        page_token = response.get('nextPageToken')
        if not page_token:
            return found

def download_drive_file(file_id, destination):
    request = drive_api.files().get_media(fileId=file_id, supportsAllDrives=True)
    with open(destination, 'wb') as handle:
        downloader = MediaIoBaseDownload(handle, request, chunksize=8 * 1024 * 1024)
        done = False
        while not done:
            status, done = downloader.next_chunk(num_retries=5)
            if status:
                print(f'{Path(destination).name}: {status.progress() * 100:.0f}%')

project_id = child_folder_id('root', 'AppleCapstoneRelighting')
results_id = child_folder_id(project_id, 'results')
run_id = child_folder_id(results_id, 'fastvit_t8_fpn_depth_normals_local50000_seed42_v1')
source_id = child_folder_id(child_folder_id(child_folder_id(project_id, 'dataset'), 'sources'), 'SynthHuman')
chunk_id = child_folder_id(source_id, CHUNK_NAME)

run_files = find_named_files(run_id, ['best_checkpoint.pt', 'config.yaml', 'raw_dataset_manifest.jsonl'])
CHECKPOINT_LOCAL = LOCAL_DATA / 'best_checkpoint.pt'
CONFIG_LOCAL = LOCAL_DATA / 'config.yaml'
MANIFEST_LOCAL = LOCAL_DATA / 'raw_dataset_manifest.jsonl'
for local_path, name in ((CHECKPOINT_LOCAL, 'best_checkpoint.pt'), (CONFIG_LOCAL, 'config.yaml'), (MANIFEST_LOCAL, 'raw_dataset_manifest.jsonl')):
    download_drive_file(run_files[name]['id'], local_path)

run_rows = [json.loads(line) for line in MANIFEST_LOCAL.read_text(encoding='utf-8').splitlines() if line.strip()]
used_ids = {int(row['subject_id'].rsplit('_', 1)[1]) for row in run_rows}
chunk_files = list_chunk_files(chunk_id)
modalities_by_id = {}
pattern = re.compile(r'^(rgb|alpha|depth|normal)_(\d+)\.(png|exr)$')
for filename, metadata in chunk_files.items():
    match = pattern.fullmatch(filename)
    if match:
        kind, raw_id, _ = match.groups()
        modalities_by_id.setdefault(int(raw_id), {})[kind] = metadata
complete_ids = sorted(sample_id for sample_id, modalities in modalities_by_id.items()
                      if {'rgb', 'alpha', 'depth', 'normal'} <= modalities.keys())
SAMPLE_IDS = [sample_id for sample_id in complete_ids if sample_id not in used_ids][:5]
if len(SAMPLE_IDS) != 5:
    raise RuntimeError(f'Found only {len(SAMPLE_IDS)} complete samples outside this run manifest in {CHUNK_NAME}.')
if set(SAMPLE_IDS) & used_ids:
    raise AssertionError('Sample selection overlaps the run manifest.')

selected = []
for sample_id in SAMPLE_IDS:
    row = {'subject_id': f'sample_{sample_id:07d}'}
    # The Drive listing uses source modality names (rgb/alpha), while the
    # dataset reader expects row keys image/mask. Map them explicitly.
    file_specs = {
        'image': ('rgb', f'rgb_{sample_id:07d}.png'),
        'mask': ('alpha', f'alpha_{sample_id:07d}.png'),
        'depth': ('depth', f'depth_{sample_id:07d}.exr'),
        'normal': ('normal', f'normal_{sample_id:07d}.exr'),
    }
    available = modalities_by_id[sample_id]
    missing = [source_key for source_key, _ in file_specs.values() if source_key not in available]
    if missing:
        raise FileNotFoundError(f'{sample_id}: missing Drive modalities {missing}')
    for row_key, (source_key, filename) in file_specs.items():
        local_path = LOCAL_DATA / filename
        download_drive_file(available[source_key]['id'], local_path)
        row[row_key] = str(local_path)
    row['depth_scale_to_m'] = 0.01
    selected.append(row)
print(f'Run manifest IDs: {len(used_ids):,}; selected held-out sample IDs: {SAMPLE_IDS}')
print('Verified: selected IDs do not occur in this run manifest (train/val/test).')


## Load checkpoint and run inference

Preprocessing is reused from the training code: RGB ImageNet normalization, 256×256 resize, EXR depth conversion to metres, alpha mask, and unit-length normals.

In [ ]:
config = yaml.safe_load(CONFIG_LOCAL.read_text(encoding='utf-8'))
exp = config['experiment']
checkpoint = torch.load(CHECKPOINT_LOCAL, map_location='cpu', weights_only=False)
state = checkpoint['model'] if isinstance(checkpoint, dict) and 'model' in checkpoint else checkpoint
model = PortraitGeometryModel(exp['backbone'], pretrained=False, fpn_width=int(exp['fpn_width']), fpn_fusion=exp.get('fpn_fusion', 'add'))
model.load_state_dict(state, strict=True)
model.to(DEVICE).eval()

dataset = RawSynthHumanDataset(selected, input_size=tuple(exp['input_size']), augment=False)
preds = []
with torch.inference_mode():
    for i in range(len(dataset)):
        x, depth_gt, normal_gt, mask = dataset[i]
        depth_pred, normal_pred = model(x.unsqueeze(0).to(DEVICE))
        preds.append({'x': x, 'depth_gt': depth_gt, 'normal_gt': normal_gt, 'mask': mask, 'depth_pred': depth_pred[0].cpu(), 'normal_pred': normal_pred[0].cpu()})
print(f"Checkpoint loaded; best epoch={checkpoint.get('best_epoch', 'unknown')}; inferred {len(preds)} samples.")

## Perbandingan checkpoint base vs fine-tune (input dan visualisasi sama)
Cell ini mengambil checkpoint fine-tune dari folder `results` di Drive dan membandingkannya dengan checkpoint base yang sudah dimuat sebelumnya. Keduanya diuji pada lima sample yang sama dengan preprocessing serta resolusi **384×512** yang sama. Dengan begitu perbedaan terutama menunjukkan dampak bobot fine-tuning, bukan perbedaan resolusi input. Hasilnya berupa dua matriks visual (depth dan normal) serta tabel metrik per gambar.

In [ ]:
# Ambil fine-tune checkpoint yang baru disalin ke Drive, lalu bandingkan pada input identik.
import pandas as pd
FINETUNE_RUN_NAME = 'fastvit_t8_fpn_depth_normals_local50000_seed42_finetune_lr2e6_384x512_100ep_v1'
ft_run_id = child_folder_id(results_id, FINETUNE_RUN_NAME)
ft_files = find_named_files(ft_run_id, ['best_checkpoint.pt', 'config.yaml'])
FT_CHECKPOINT_LOCAL = LOCAL_DATA / 'finetune_best_checkpoint.pt'
FT_CONFIG_LOCAL = LOCAL_DATA / 'finetune_config.yaml'
for local_path, filename in ((FT_CHECKPOINT_LOCAL, 'best_checkpoint.pt'), (FT_CONFIG_LOCAL, 'config.yaml')):
    download_drive_file(ft_files[filename]['id'], local_path)

ft_config = yaml.safe_load(FT_CONFIG_LOCAL.read_text(encoding='utf-8'))
ft_exp = ft_config['experiment']
base_exp = exp
for key in ('backbone', 'fpn_width', 'depth_mode'):
    if base_exp.get(key) != ft_exp.get(key):
        raise ValueError(f'Checkpoint architecture/config mismatch for {key}: {base_exp.get(key)} vs {ft_exp.get(key)}')

# Cell sebelumnya sudah memuat model base. Buat model fine-tune terpisah.
model_base = model.eval()
ft_state = torch.load(FT_CHECKPOINT_LOCAL, map_location='cpu', weights_only=False)
model_ft = PortraitGeometryModel(ft_exp['backbone'], pretrained=False, fpn_width=int(ft_exp['fpn_width']), fpn_fusion=ft_exp.get('fpn_fusion', 'add'))
model_ft.load_state_dict(ft_state['model'], strict=True)
model_ft.to(DEVICE).eval()

# Ukuran sama untuk kedua model agar perbandingan checkpoint terkontrol.
COMPARE_INPUT_SIZE = tuple(int(v) for v in ft_exp['input_size'])  # (H, W), expected 384×512
compare_dataset = RawSynthHumanDataset(selected, input_size=COMPARE_INPUT_SIZE, augment=False)
comparison = []
with torch.inference_mode():
    for i, row in enumerate(selected):
        x, depth_gt, normal_gt, mask = compare_dataset[i]
        xb = x.unsqueeze(0).to(DEVICE)
        db, nb = model_base(xb)
        df, nf = model_ft(xb)
        comparison.append({
            'row': row, 'x': x.cpu(), 'depth_gt': depth_gt[0].cpu(),
            'normal_gt': normal_gt.cpu(), 'mask': mask[0].bool().cpu(),
            'depth_base': db[0, 0].float().cpu(), 'normal_base': nb[0].float().cpu(),
            'depth_ft': df[0, 0].float().cpu(), 'normal_ft': nf[0].float().cpu(),
        })
print(f'Base best epoch={checkpoint.get("best_epoch", "unknown")} | fine-tune best epoch={ft_state.get("best_epoch", "unknown")}')
print('Same held-out samples:', [item['row']['subject_id'] for item in comparison])
print('Same input resolution for both models:', COMPARE_INPUT_SIZE)

# Metrik per sample. MAE/RMSE dalam cm; skala prediksi/GT hanya indikator bias skala.
metric_rows = []
for item in comparison:
    m = item['mask']
    gt = item['depth_gt']; gt_n = F.normalize(item['normal_gt'], dim=0, eps=1e-6)
    rec = {'sample_id': item['row']['subject_id'], 'GT_depth_median_m': float(gt[m].median())}
    for label in ('base', 'ft'):
        dp = item[f'depth_{label}']; npred = F.normalize(item[f'normal_{label}'], dim=0, eps=1e-6)
        de = dp[m] - gt[m]
        angle = torch.rad2deg(torch.acos((gt_n * npred).sum(0).clamp(-1, 1)))[m]
        rec[f'{label}_depth_MAE_cm'] = float(de.abs().mean()*100)
        rec[f'{label}_depth_RMSE_cm'] = float(de.square().mean().sqrt()*100)
        rec[f'{label}_pred_GT_median_ratio'] = float(dp[m].median() / gt[m].median().clamp_min(1e-8))
        rec[f'{label}_normal_mean_deg'] = float(angle.mean())
        rec[f'{label}_normal_median_deg'] = float(angle.median())
        rec[f'{label}_normal_within_11_25_pct'] = float((angle < 11.25).float().mean()*100)
    metric_rows.append(rec)
metric_table = pd.DataFrame(metric_rows)
display(metric_table.round(3))
print('Rata-rata per-image (tiap gambar bobot sama):')
display(metric_table.drop(columns='sample_id').mean().to_frame('mean_per_image').round(3))

# Matriks depth: rentang warna depth sama untuk seluruh gambar dan kedua checkpoint.
all_depth = np.concatenate([item['depth_gt'].numpy()[item['mask'].numpy()] for item in comparison] +
                           [item['depth_base'].numpy()[item['mask'].numpy()] for item in comparison] +
                           [item['depth_ft'].numpy()[item['mask'].numpy()] for item in comparison])
depth_lo, depth_hi = np.percentile(all_depth[np.isfinite(all_depth)], [2, 98])
all_errors = np.concatenate([np.abs(item[f'depth_{key}'].numpy()[item['mask'].numpy()] - item['depth_gt'].numpy()[item['mask'].numpy()])
                             for item in comparison for key in ('base', 'ft')])
err_hi = max(float(np.percentile(all_errors[np.isfinite(all_errors)], 98)), 1e-6)
fig, axes = plt.subplots(len(comparison), 6, figsize=(22, 4*len(comparison)), squeeze=False)
headers = ['RGB', 'Depth GT (m)', 'Base depth (m)', 'Fine-tune depth (m)', 'Base |error| (m)', 'Fine-tune |error| (m)']
for j, title in enumerate(headers): axes[0, j].set_title(title)
for i, item in enumerate(comparison):
    m = item['mask'].numpy(); gt=item['depth_gt'].numpy()
    rgb = item['x'].permute(1,2,0).numpy()*np.array([0.229,0.224,0.225])+np.array([0.485,0.456,0.406])
    arrays = [rgb, gt, item['depth_base'].numpy(), item['depth_ft'].numpy(),
              np.abs(item['depth_base'].numpy()-gt), np.abs(item['depth_ft'].numpy()-gt)]
    for j, arr in enumerate(arrays):
        ax=axes[i,j]
        if j == 0: ax.imshow(np.clip(arr,0,1))
        elif j in (1,2,3): ax.imshow(np.ma.masked_where(~m,arr),cmap='viridis',vmin=depth_lo,vmax=depth_hi)
        else: ax.imshow(np.ma.masked_where(~m,arr),cmap='inferno',vmin=0,vmax=err_hi)
        ax.axis('off')
    axes[i,0].set_ylabel(item['row']['subject_id'], fontsize=9); axes[i,0].yaxis.set_visible(True); axes[i,0].tick_params(left=False,labelleft=False)
fig.suptitle('Depth: base vs fine-tune — shared color scales'); fig.tight_layout()
depth_compare_png = LOCAL_DATA / 'base_vs_finetune_depth_384x512.png'
fig.savefig(depth_compare_png, dpi=140, bbox_inches='tight')
print(f'Shared depth scale: {depth_lo:.3f}–{depth_hi:.3f} m | shared error scale: 0–{err_hi:.3f} m')
plt.show()

# Matriks normal: XYZ memakai encoding warna yang sama; error sudut memakai skala 0–90°.
fig, axes = plt.subplots(len(comparison), 6, figsize=(22, 4*len(comparison)), squeeze=False)
headers = ['RGB', 'Normal GT (XYZ→RGB)', 'Base normal', 'Fine-tune normal', 'Base angular error (°)', 'Fine-tune angular error (°)']
for j, title in enumerate(headers): axes[0,j].set_title(title)
for i, item in enumerate(comparison):
    m=item['mask'].numpy(); rgb=item['x'].permute(1,2,0).numpy()*np.array([0.229,0.224,0.225])+np.array([0.485,0.456,0.406])
    ng=F.normalize(item['normal_gt'],dim=0,eps=1e-6)
    panels=[rgb]
    for key in ('normal_gt','normal_base','normal_ft'):
        nv=F.normalize(item[key],dim=0,eps=1e-6).numpy()
        color=np.moveaxis(np.clip(nv,-1,1)*0.5+0.5,0,-1); color[~m]=0; panels.append(color)
    for key in ('normal_base','normal_ft'):
        nv=F.normalize(item[key],dim=0,eps=1e-6)
        angle=torch.rad2deg(torch.acos((ng*nv).sum(0).clamp(-1,1))).numpy(); angle[~m]=np.nan; panels.append(angle)
    for j,arr in enumerate(panels):
        ax=axes[i,j]
        if j==0 or j in (1,2,3): ax.imshow(np.clip(arr,0,1))
        else: ax.imshow(arr,cmap='inferno',vmin=0,vmax=90)
        ax.axis('off')
    axes[i,0].set_ylabel(item['row']['subject_id'],fontsize=9); axes[i,0].yaxis.set_visible(True); axes[i,0].tick_params(left=False,labelleft=False)
fig.suptitle('Normals: same XYZ color encoding; angular error shared at 0–90°'); fig.tight_layout()
normal_compare_png = LOCAL_DATA / 'base_vs_finetune_normals_384x512.png'
fig.savefig(normal_compare_png, dpi=140, bbox_inches='tight')
plt.show()
compare_json = LOCAL_DATA / 'base_vs_finetune_384x512_metrics.json'
compare_json.write_text(json.dumps(metric_rows, indent=2), encoding='utf-8')
print('Metrics saved:', compare_json)
print('Depth figure path:', depth_compare_png)
print('Normal figure path:', normal_compare_png)
print('Interpretasi: bandingkan MAE/RMSE dan normal error di tabel; jangan menyimpulkan hanya dari colormap. Nilai lebih kecil berarti error lebih kecil.')

## Visual comparison dan metrik pada lima sampel luar manifest

Depth GT dan prediksi memakai skala warna yang sama per sampel. Normal GT/prediksi menampilkan XYZ sebagai RGB; penilaian numeriknya memakai sudut antarvektor. Semua metrik dihitung hanya pada piksel valid di mask alpha SynthHuman. Ringkasan agregat adalah rata-rata lima metrik per gambar.


In [ ]:
metrics = []
all_abs_depth_errors = []
all_squared_depth_errors = []
all_valid_gt_depths = []
fig, axes = plt.subplots(len(selected), 5, figsize=(19, 4 * len(selected)), squeeze=False)
for j, title in enumerate(['Input RGB', 'Depth GT (m)', 'Depth prediction (m)', 'Normals GT', 'Normals prediction']):
    axes[0, j].set_title(title)
for i, (row, out) in enumerate(zip(selected, preds)):
    mask = out['mask'][0].bool().numpy()
    dgt, dp = out['depth_gt'][0].numpy(), out['depth_pred'][0].numpy()
    ngt, npred = out['normal_gt'].numpy(), out['normal_pred'].numpy()
    abs_error = np.abs(dp[mask] - dgt[mask])
    mae = float(abs_error.mean())
    rmse = float(np.sqrt(((dp[mask] - dgt[mask]) ** 2).mean()))
    gt_median = float(np.median(dgt[mask]))
    mae_over_gt_median_pct = 100.0 * mae / max(gt_median, 1e-8)
    all_abs_depth_errors.append(abs_error)
    all_squared_depth_errors.append((dp[mask] - dgt[mask]) ** 2)
    all_valid_gt_depths.append(dgt[mask])
    a = F.normalize(torch.from_numpy(ngt), dim=0, eps=1e-6)
    b = F.normalize(torch.from_numpy(npred), dim=0, eps=1e-6)
    angles = np.degrees(np.arccos((a * b).sum(0).clamp(-1, 1).numpy()[mask]))
    nmean, nmedian = float(angles.mean()), float(np.median(angles))
    within_11_25 = float((angles < 11.25).mean())
    metrics.append({
        'sample_id': row['subject_id'],
        'depth_gt_median_m': gt_median,
        'depth_mae_m': mae,
        'depth_mae_over_gt_median_pct': mae_over_gt_median_pct,
        'depth_rmse_m': rmse,
        'normal_mean_deg': nmean,
        'normal_median_deg': nmedian,
        'normal_within_11_25_deg': within_11_25,
    })
    rgb = out['x'].permute(1, 2, 0).numpy() * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
    lo, hi = np.percentile(dgt[mask], [2, 98])
    ngt_rgb = np.moveaxis(np.clip(ngt, -1, 1) * 0.5 + 0.5, 0, -1)
    np_rgb = np.moveaxis(np.clip(npred, -1, 1) * 0.5 + 0.5, 0, -1)
    ngt_rgb[~mask] = 0
    np_rgb[~mask] = 0
    panels = [np.clip(rgb, 0, 1), np.ma.masked_where(~mask, dgt), np.ma.masked_where(~mask, dp), ngt_rgb, np_rgb]
    for j, panel in enumerate(panels):
        ax = axes[i, j]
        if j in (1, 2):
            ax.imshow(panel, cmap='viridis', vmin=lo, vmax=hi)
        else:
            ax.imshow(panel)
        ax.axis('off')
    axes[i, 0].set_ylabel(
        f"{row['subject_id']}\nGT median {gt_median:.2f} m | MAE {mae*100:.1f} cm\nMAE / GT median {mae_over_gt_median_pct:.1f}% | Normal {nmean:.1f}°",
        fontsize=9,
    )
    axes[i, 0].yaxis.set_visible(True)
    axes[i, 0].tick_params(left=False, labelleft=False)

pooled_gt_median = float(np.median(np.concatenate(all_valid_gt_depths)))
pooled_mae = float(np.mean(np.concatenate(all_abs_depth_errors)))
pooled_rmse = float(np.sqrt(np.mean(np.concatenate(all_squared_depth_errors))))
summary = {
    key: float(np.mean([item[key] for item in metrics]))
    for key in ('depth_gt_median_m', 'depth_mae_m', 'depth_mae_over_gt_median_pct', 'depth_rmse_m', 'normal_mean_deg', 'normal_median_deg', 'normal_within_11_25_deg')
}
summary.update({
    'pooled_valid_pixel_gt_median_m': pooled_gt_median,
    'pooled_valid_pixel_depth_mae_m': pooled_mae,
    'pooled_valid_pixel_depth_rmse_m': pooled_rmse,
    'pooled_mae_over_gt_median_pct': 100.0 * pooled_mae / max(pooled_gt_median, 1e-8),
    'aggregation': 'per-image metrics are averaged equally; pooled depth values combine valid pixels from all five images',
    'interpretation': 'MAE divided by median GT depth is a relative-error scale, not an official accuracy percentage',
})
fig.suptitle('FastViT-T8 + FPN — five SynthHuman samples outside the 50k run manifest', y=1.002, fontsize=15)
fig.tight_layout()
out_png = LOCAL_DATA / 'colab_5_heldout_visualization.png'
out_json = LOCAL_DATA / 'colab_5_heldout_metrics.json'
fig.savefig(out_png, dpi=160, bbox_inches='tight')
out_json.write_text(json.dumps({'per_image': metrics, 'summary': summary}, indent=2), encoding='utf-8')
plt.show()
print('Depth scale + metrics for five images:', json.dumps(summary, indent=2))
print('Per-image metrics:', json.dumps(metrics, indent=2))
print('Note: MAE / median GT depth is relative error, not a standard accuracy percentage.')
print('Saved in Colab runtime:', out_png, 'and', out_json)
from google.colab import files
files.download(str(out_png))
files.download(str(out_json))


## Inspeksi fitur: FastViT, operasi internal FPN, dan upsampling head

Cell ini memakai sampel pertama dari lima sampel held-out. Ia menampilkan empat keluaran fitur FastViT, lalu untuk setiap level FPN memvisualisasikan lateral feature, bilinear upsample dari level yang lebih kasar, hasil fusion (add atau concat sesuai konfigurasi), dan smooth block. Peta terakhir menunjukkan output head sebelum dan sesudah interpolasi akhir serta error terhadap GT. Semua ukuran tensor dicetak.

Peta aktivasi adalah rata-rata nilai absolut antar-channel untuk melihat lokasi respons, bukan ground truth fitur. Skala warnanya dinormalisasi per peta; gunakan ukuran tensor, jalur operasi, dan peta error final bersama-sama.


In [ ]:
# Trace the actual high-level stages and each top-down FPN operation.
sample_index = 0
out = preds[sample_index]
x_inspect = out['x'].unsqueeze(0).to(DEVICE)
input_hw = tuple(x_inspect.shape[-2:])
model.eval()

with torch.inference_mode():
    # FastViT features: coarse-to-fine list returned by timm's features_only interface.
    encoder_levels = list(model.encoder(x_inspect))

    # FPN lateral projections at each encoder resolution.
    lateral_features = [layer(feat) for layer, feat in zip(model.neck.lateral, encoder_levels)]
    fpn_trace = {}

    # P5 starts from the coarsest level; then top-down bilinear resize + skip addition + smoothing.
    top_down = model.neck.smooth[-1](lateral_features[-1])
    fpn_trace['P5'] = {
        'lateral': lateral_features[-1],
        'upsample': None,
        'merge': None,
        'smooth': top_down,
    }
    for index in range(len(encoder_levels) - 2, -1, -1):
        level_name = f'P{index + 2}'
        upsampled = F.interpolate(
            top_down,
            size=encoder_levels[index].shape[-2:],
            mode='bilinear',
            align_corners=False,
        )
        if model.neck.fusion == 'add':
            merged = upsampled + lateral_features[index]
        else:
            merged = model.neck.fusions[index](torch.cat((upsampled, lateral_features[index]), dim=1))
        top_down = model.neck.smooth[index](merged)
        fpn_trace[level_name] = {
            'lateral': lateral_features[index],
            'upsample': upsampled,
            'merge': merged,
            'smooth': top_down,
        }
    fused = top_down

    # Follow both task heads exactly, exposing their final bilinear upsample.
    depth_refined = model.depth_head.refine(fused)
    depth_raw = model.depth_head.predict(depth_refined)
    depth_upsampled = F.interpolate(depth_raw, size=input_hw, mode='bilinear', align_corners=False)
    depth_manual = F.softplus(depth_upsampled)

    normal_refined = model.normal_head.refine(fused)
    normal_raw = model.normal_head.predict(normal_refined)
    normal_upsampled = F.interpolate(normal_raw, size=input_hw, mode='bilinear', align_corners=False)
    normal_manual = F.normalize(normal_upsampled, p=2, dim=1, eps=1e-6)

    # Compare traced reconstruction with the model's normal forward path.
    depth_forward, normal_forward = model(x_inspect)
    depth_trace_delta = (depth_manual - depth_forward).abs().max().item()
    normal_trace_delta = (normal_manual - normal_forward).abs().max().item()

print('Input:', tuple(x_inspect.shape))
for level, feat in enumerate(encoder_levels, start=1):
    print(f'FastViT feature level {level}: {tuple(feat.shape)}')
for level_name in ('P5', 'P4', 'P3', 'P2'):
    items = fpn_trace[level_name]
    print(f'{level_name}: lateral={tuple(items["lateral"].shape)}; '
          f'upsample={None if items["upsample"] is None else tuple(items["upsample"].shape)}; '
          f'merge={None if items["merge"] is None else tuple(items["merge"].shape)}; '
          f'smooth={tuple(items["smooth"].shape)}')
print('Depth head:', f'refined={tuple(depth_refined.shape)} -> raw={tuple(depth_raw.shape)} -> bilinear={tuple(depth_upsampled.shape)} -> softplus={tuple(depth_manual.shape)}')
print('Normal head:', f'refined={tuple(normal_refined.shape)} -> raw={tuple(normal_raw.shape)} -> bilinear={tuple(normal_upsampled.shape)} -> unit-normal={tuple(normal_manual.shape)}')
print(f'Trace matches model forward: max abs diff depth={depth_trace_delta:.3g}, normal={normal_trace_delta:.3g}')

# Activation strength = mean absolute response across channels; resize only for visualization.
def activation_map(tensor):
    value = tensor.detach().float().abs().mean(dim=1, keepdim=True)
    value = F.interpolate(value, size=input_hw, mode='bilinear', align_corners=False)
    return value[0, 0].cpu().numpy()

def show_activation(ax, tensor, title):
    image = activation_map(tensor)
    lo, hi = np.nanpercentile(image, [2, 98])
    artist = ax.imshow(image, cmap='magma', vmin=lo, vmax=max(hi, lo + 1e-8))
    ax.set_title(f'{title}\nraw {tuple(tensor.shape[-2:])}', fontsize=8)
    ax.axis('off')
    return artist

# Figure A: FastViT multiscale outputs.
fig, axes = plt.subplots(1, len(encoder_levels), figsize=(4 * len(encoder_levels), 4), squeeze=False)
for index, feat in enumerate(encoder_levels):
    show_activation(axes[0, index], feat, f'FastViT level {index+1}')
fig.suptitle('FastViT feature pyramid outputs', y=1.02)
fig.tight_layout()
plt.show()

# Figure B: every FPN level and each operation in its top-down path.
operation_columns = ('lateral', 'upsample', 'merge', 'smooth')
fig, axes = plt.subplots(4, 4, figsize=(16, 14), squeeze=False)
for row, level_name in enumerate(('P5', 'P4', 'P3', 'P2')):
    for col, operation in enumerate(operation_columns):
        tensor = fpn_trace[level_name][operation]
        ax = axes[row, col]
        if tensor is None:
            ax.axis('off')
            continue
        show_activation(ax, tensor, f'{level_name}: {operation}')
        if row == 0:
            ax.set_title(f'{operation}\nraw {tuple(tensor.shape[-2:])}', fontsize=8)
    axes[row, 0].set_ylabel(level_name, fontsize=11)
fig.suptitle(f'FPN internals ({model.neck.fusion}): lateral → bilinear upsample → merge → smooth', y=1.01)
fig.tight_layout()
plt.show()

# Figure C: coarse head outputs, final interpolation, and final errors against GT.
mask = out['mask'][0].bool().numpy()
depth_gt = out['depth_gt'][0].numpy()
depth_pred = depth_forward[0, 0].detach().float().cpu().numpy()
normal_gt = F.normalize(out['normal_gt'].unsqueeze(0).float(), dim=1, eps=1e-6)
normal_pred = F.normalize(normal_forward.detach().float().cpu(), dim=1, eps=1e-6)
normal_error = torch.rad2deg(torch.acos((normal_gt * normal_pred).sum(1).clamp(-1, 1)))[0].numpy()
normal_error[~mask] = np.nan
depth_error = np.abs(depth_pred - depth_gt)
depth_error[~mask] = np.nan
depth_lo, depth_hi = np.percentile(depth_gt[mask], [2, 98])
fig, axes = plt.subplots(2, 4, figsize=(16, 8), squeeze=False)
show_activation(axes[0, 0], depth_raw, 'Depth raw head')
show_activation(axes[0, 1], depth_upsampled, 'Depth after final bilinear upsample')
show_activation(axes[0, 2], normal_raw, 'Normal raw head')
show_activation(axes[0, 3], normal_upsampled, 'Normal after final bilinear upsample')
axes[1, 0].imshow(np.ma.masked_where(~mask, depth_gt), cmap='viridis', vmin=depth_lo, vmax=depth_hi)
axes[1, 0].set_title('GT depth (m)'); axes[1, 0].axis('off')
axes[1, 1].imshow(np.ma.masked_where(~mask, depth_pred), cmap='viridis', vmin=depth_lo, vmax=depth_hi)
axes[1, 1].set_title('Predicted depth (m)'); axes[1, 1].axis('off')
axes[1, 2].imshow(np.ma.masked_where(~mask, depth_error), cmap='inferno', vmin=0, vmax=np.nanpercentile(depth_error, 98))
axes[1, 2].set_title('Depth absolute error (m)'); axes[1, 2].axis('off')
axes[1, 3].imshow(normal_error, cmap='inferno', vmin=0, vmax=90)
axes[1, 3].set_title('Normal angular error (°)'); axes[1, 3].axis('off')
fig.suptitle(f"Head reconstruction + GT errors — {selected[sample_index]['subject_id']}", y=1.02)
fig.tight_layout()
plt.show()
print('Activation maps show response strength, not correctness. Error maps show final output mismatch with GT.')


## Bedah detail encoder → FPN decoder → task heads
Jalankan setelah cell **Inspeksi fitur: FastViT, operasi internal FPN, dan upsampling head**. Model kita tidak memiliki decoder U-Net terpisah: jalur top-down FPN berperan sebagai decoder, lalu depth/normal head melakukan refine konvolusi dan proyeksi 1×1. Resolusi penuh dibuat dengan bilinear interpolation, bukan transposed convolution. Cell ini mengukur kontribusi lateral vs top-down, perubahan akibat smoothing, gradien fitur/output, dan contoh channel individual agar dugaan bottleneck decoder bisa diuji lebih terarah. Peta aktivasi tetap bukan peta “kebenaran” model.


In [ ]:
# Analisis lanjutan dari trace pada cell sebelumnya; model/checkpoint tidak diubah.
import pandas as pd

def _rms(t):
    return float(t.detach().float().square().mean().sqrt().cpu())

def _mean_abs(t):
    return float(t.detach().float().abs().mean().cpu())

def _cosine_flat(a, b):
    a = a.detach().float().reshape(-1)
    b = b.detach().float().reshape(-1)
    return float(F.cosine_similarity(a[None], b[None]).cpu())

def _hf_energy(t):
    # Rata-rata perubahan absolut antar-piksel tetangga, dihitung pada resolusi tensor asli.
    t = t.detach().float()
    dx = (t[..., :, 1:] - t[..., :, :-1]).abs().mean() if t.shape[-1] > 1 else t.new_tensor(0.)
    dy = (t[..., 1:, :] - t[..., :-1, :]).abs().mean() if t.shape[-2] > 1 else t.new_tensor(0.)
    return float(((dx + dy) * 0.5).cpu())

# 1) Ringkasan ukuran, aktivasi, dan detail spasial pada setiap tahap.
layer_rows = []
for i, feat in enumerate(encoder_levels):
    layer_rows.append({
        'stage': f'FastViT S{i+1}', 'shape_CxHxW': str(tuple(feat.shape[1:])),
        'stride_vs_input': f'{input_hw[0]/feat.shape[-2]:.1f}x{input_hw[1]/feat.shape[-1]:.1f}',
        'channels': int(feat.shape[1]), 'RMS': _rms(feat),
        'mean_abs': _mean_abs(feat), 'spatial_gradient_energy': _hf_energy(feat),
    })

for name in ('P5', 'P4', 'P3', 'P2'):
    trace = fpn_trace[name]
    lat, up, merge, smooth = trace['lateral'], trace['upsample'], trace['merge'], trace['smooth']
    row = {
        'stage': f'FPN {name}', 'shape_CxHxW': str(tuple(smooth.shape[1:])),
        'stride_vs_input': f'{input_hw[0]/smooth.shape[-2]:.1f}x{input_hw[1]/smooth.shape[-1]:.1f}',
        'channels': int(smooth.shape[1]), 'RMS': _rms(smooth),
        'mean_abs': _mean_abs(smooth), 'spatial_gradient_energy': _hf_energy(smooth),
    }
    if up is not None:
        row.update({
            'lateral_RMS': _rms(lat), 'topdown_RMS': _rms(up),
            'topdown/lateral_RMS': _rms(up) / max(_rms(lat), 1e-12),
            'cosine(topdown,lateral)': _cosine_flat(up, lat),
            'merge_RMS': _rms(merge),
            'smooth_delta/merge_RMS': _rms(smooth - merge) / max(_rms(merge), 1e-12),
            'HF_after/before_smooth': _hf_energy(smooth) / max(_hf_energy(merge), 1e-12),
        })
    else:
        row.update({'lateral_RMS': _rms(lat), 'topdown_RMS': float('nan'),
                    'topdown/lateral_RMS': float('nan'), 'cosine(topdown,lateral)': float('nan'),
                    'merge_RMS': float('nan'), 'smooth_delta/merge_RMS': float('nan'),
                    'HF_after/before_smooth': float('nan')})
    layer_rows.append(row)

for name, tensor in [
    ('FPN fused', fused), ('Depth refine', depth_refined), ('Depth raw logits', depth_raw),
    ('Normal refine', normal_refined), ('Normal raw logits', normal_raw),
]:
    layer_rows.append({
        'stage': name, 'shape_CxHxW': str(tuple(tensor.shape[1:])),
        'stride_vs_input': f'{input_hw[0]/tensor.shape[-2]:.1f}x{input_hw[1]/tensor.shape[-1]:.1f}',
        'channels': int(tensor.shape[1]), 'RMS': _rms(tensor),
        'mean_abs': _mean_abs(tensor), 'spatial_gradient_energy': _hf_energy(tensor),
    })
layer_detail_table = pd.DataFrame(layer_rows)
print('Tahap encoder/decoder/head: ukuran dan kekuatan respons pada resolusi asli tiap tensor.')
display(layer_detail_table.round(4))
print('Interpretasi FPN: topdown/lateral_RMS menunjukkan besar relatif respons; cosine mengukur kemiripan arah fitur; HF_after/before_smooth < 1 berarti smoothing mengurangi perubahan spasial.')

# 2) Lihat kontribusi top-down dan perubahan smooth block secara eksplisit.
def _show_feature_map(ax, tensor, title, cmap='magma'):
    v = tensor.detach().float().abs().mean(dim=1)[0].cpu().numpy()
    lo, hi = np.percentile(v[np.isfinite(v)], [2, 98])
    ax.imshow(v, cmap=cmap, vmin=lo, vmax=max(hi, lo + 1e-8), interpolation='nearest')
    ax.set_title(f'{title}\nraw {v.shape[0]}×{v.shape[1]}', fontsize=8)
    ax.axis('off')

fig, axes = plt.subplots(4, 6, figsize=(21, 13), squeeze=False)
for r, name in enumerate(('P5', 'P4', 'P3', 'P2')):
    tr = fpn_trace[name]
    tensors = [tr['lateral'], tr['upsample'], tr['merge'], tr['smooth'],
               None if tr['upsample'] is None else tr['merge'] - tr['lateral'],
               None if tr['merge'] is None else tr['smooth'] - tr['merge']]
    titles = ['lateral lokal', 'top-down upsample', 'hasil fusion', 'hasil smooth',
              'fusion output - lateral', 'smooth change']
    for c, (tensor, title) in enumerate(zip(tensors, titles)):
        ax=axes[r,c]
        if tensor is None:
            ax.axis('off'); continue
        _show_feature_map(ax, tensor, f'{name}: {title}')
    axes[r,0].set_ylabel(name, fontsize=11)
fig.suptitle('FPN: respons lokal, informasi top-down, penggabungan, dan efek smoothing', y=1.01)
fig.tight_layout(); plt.show()

# 3) Channel individual terpilih berdasarkan variasi spasial, bukan indeks yang dipilih manual.
def _show_top_channels(axrow, tensor, label, count=6):
    fmap = tensor.detach().float()[0].cpu()  # C,H,W
    scores = fmap.flatten(1).std(dim=1)
    ids = torch.argsort(scores, descending=True)[:min(count, fmap.shape[0])].tolist()
    for ax, channel in zip(axrow, ids):
        img=fmap[channel].numpy()
        lim=float(np.percentile(np.abs(img), 98))
        ax.imshow(img, cmap='RdBu_r', vmin=-max(lim,1e-8), vmax=max(lim,1e-8), interpolation='nearest')
        ax.set_title(f'{label} ch {channel}\nstd={scores[channel].item():.3g}', fontsize=8)
        ax.axis('off')
    for ax in axrow[len(ids):]: ax.axis('off')

fig, axes = plt.subplots(3, 6, figsize=(18, 9), squeeze=False)
_show_top_channels(axes[0], fused, 'FPN fused')
_show_top_channels(axes[1], depth_refined, 'Depth refine')
_show_top_channels(axes[2], normal_refined, 'Normal refine')
fig.suptitle('Channel dengan variasi spasial terbesar pada decoder dan task heads', y=1.01)
fig.tight_layout(); plt.show()

# 4) Bandingkan apakah keluaran akhir terlalu halus terhadap GT di area interior.
# Metrik gradien sederhana ini melengkapi MAE; ia bukan pengganti metrik utama.
_diag_mask = torch.as_tensor(out['mask'][0]).bool().cpu()
_dgt = torch.as_tensor(out['depth_gt'][0]).float().cpu()
_dpred = depth_forward[0,0].detach().float().cpu()
_ngt = F.normalize(torch.as_tensor(out['normal_gt']).float().cpu(), dim=0, eps=1e-6)
_npred = F.normalize(normal_forward[0].detach().float().cpu(), dim=0, eps=1e-6)
_eroded = (1.0 - F.max_pool2d(1.0 - _diag_mask[None,None].float(), 3, 1, 1))[0,0] > .5
_interior = _diag_mask & _eroded

def _masked_gradient_energy(tensor, mask):
    # tensor C,H,W; hanya pasangan piksel yang keduanya berada di interior mask.
    dx = (tensor[..., :, 1:] - tensor[..., :, :-1]).abs().mean(dim=0)
    dy = (tensor[..., 1:, :] - tensor[..., :-1, :]).abs().mean(dim=0)
    mx = mask[:,1:] & mask[:,:-1]
    my = mask[1:,:] & mask[:-1,:]
    vals=[]
    if mx.any(): vals.append(dx[mx].mean())
    if my.any(): vals.append(dy[my].mean())
    return float(torch.stack(vals).mean()) if vals else float('nan')

depth_gt_grad = _masked_gradient_energy(_dgt[None], _interior)
depth_pred_grad = _masked_gradient_energy(_dpred[None], _interior)
normal_gt_grad = _masked_gradient_energy(_ngt, _interior)
normal_pred_grad = _masked_gradient_energy(_npred, _interior)
print('Interior gradient energy (rasio pred/GT < 1 dapat menandakan output lebih halus, tetapi lihat visual dan error juga):')
display(pd.DataFrame([
    {'task':'depth (m/pixel)', 'GT_gradient':depth_gt_grad, 'prediction_gradient':depth_pred_grad,
     'prediction/GT':depth_pred_grad/max(depth_gt_grad,1e-12)},
    {'task':'normal (unit-vector/pixel)', 'GT_gradient':normal_gt_grad, 'prediction_gradient':normal_pred_grad,
     'prediction/GT':normal_pred_grad/max(normal_gt_grad,1e-12)},
]).round(4))


## Analisis GT sebelum dan sesudah resize
Jalankan setelah cell inspeksi FPN. Cell ini membaca **satu sample yang sama** dari Drive lalu menampilkan RGB asli, RGB pada resolusi model dengan kontur mask, depth GT setelah resize yang dipakai loader, serta normal GT setelah resize dan normalisasi. Tujuannya mengecek alignment dan detail label yang mungkin hilang saat preprocessing.


In [ ]:
# Audit satu sample dari file mentah sampai tensor resolusi model.
from PIL import Image
from relight_train.data import _read_exr, _resize_array

AUDIT_INDEX = 0
row = selected[AUDIT_INDEX]
target_h, target_w = (int(v) for v in exp['input_size'])
with Image.open(row['image']) as im:
    rgb_native = np.asarray(im.convert('RGB'))
    rgb_resized = np.asarray(im.convert('RGB').resize((target_w, target_h), Image.Resampling.BILINEAR))
with Image.open(row['mask']) as im:
    alpha_native = np.asarray(im.convert('L')) > 0
    alpha_resized = _resize_array(alpha_native.astype(np.uint8)[..., None] * 255, (target_h, target_w), True)[..., 0] > 0
raw_depth = _read_exr(Path(row['depth']), 'depth').astype(np.float32)
raw_normal = _read_exr(Path(row['normal']), 'normal').astype(np.float32)
raw_depth_resized = _resize_array(raw_depth, (target_h, target_w))[..., 0]
depth_resized_m = raw_depth_resized * float(row.get('depth_scale_to_m', 0.01))
normal_resized = _resize_array(raw_normal, (target_h, target_w))
normal_resized = F.normalize(torch.from_numpy(normal_resized).permute(2, 0, 1), dim=0, eps=1e-6).permute(1, 2, 0).numpy()
# Samakan validitas dengan RawSynthHumanDataset: cek batas dan sentinel pada nilai EXR mentah, sebelum konversi ke meter.
valid_depth_raw = (np.isfinite(raw_depth_resized) & (raw_depth_resized > 0) & (raw_depth_resized < 1000)
                   & ~np.isclose(raw_depth_resized, 65504.0, rtol=0.0, atol=1.0))
valid = alpha_resized & valid_depth_raw & np.isfinite(normal_resized).all(axis=-1)
print('Sample:', row['subject_id'])
print('Native RGB / alpha / depth / normal:', rgb_native.shape, alpha_native.shape, raw_depth.shape, raw_normal.shape)
print('Model target H×W:', (target_h, target_w), '| valid-label fraction:', f'{valid.mean():.3f}')
print('Depth source values are scaled by', row.get('depth_scale_to_m', 0.01), 'to metres.')
if valid.any():
    print('Valid GT depth p01/p50/p99 (m):', np.percentile(depth_resized_m[valid], [1, 50, 99]).round(3).tolist())

normal_rgb = np.clip(normal_resized, -1, 1) * 0.5 + 0.5
normal_rgb[~valid] = 0
fig, axes = plt.subplots(1, 5, figsize=(20, 5))
axes[0].imshow(rgb_native); axes[0].set_title(f'RGB asli\n{rgb_native.shape[1]}×{rgb_native.shape[0]}')
axes[1].imshow(rgb_resized); axes[1].contour(alpha_resized, levels=[0.5], colors='lime', linewidths=0.7); axes[1].set_title('RGB resize + batas mask')
depth_show = np.ma.masked_where(~valid, depth_resized_m)
if valid.any():
    dlo, dhi = np.percentile(depth_resized_m[valid], [2, 98])
else:
    dlo, dhi = 0, 1
im = axes[2].imshow(depth_show, cmap='viridis', vmin=dlo, vmax=dhi); axes[2].set_title('Depth GT setelah resize (m)')
fig.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)
axes[3].imshow(normal_rgb); axes[3].set_title('Normal GT setelah resize (XYZ→RGB)')
axes[4].imshow(valid, cmap='gray'); axes[4].set_title('Piksel label valid')
for ax in axes: ax.axis('off')
fig.suptitle('Audit preprocessing: periksa kontur mask, alignment, dan detail GT')
fig.tight_layout(); plt.show()
print('Catatan: normal XYZ ditampilkan sebagai warna encoding arah, bukan warna material.')


## Cek rasio aspek: resize sekarang vs mempertahankan bentuk asli
Cell ini hanya mengaudit transformasi input dan label. Saat ini model fine-tune dilatih dengan resize langsung ke ukuran target, sehingga prediksi pada bentuk yang sudah dikoreksi belum valid untuk evaluasi sampai model dilatih ulang/fine-tune dengan preprocessing yang sama.


In [ ]:
# Bandingkan resize langsung (yang dipakai training saat ini) dengan ukuran berasio asli.
source_h, source_w = rgb_native.shape[:2]
current_h, current_w = (int(v) for v in exp['input_size'])
source_ratio = source_w / source_h
current_ratio = current_w / current_h
# Pertahankan luas piksel target kurang lebih sama, tetapi samakan rasio W/H dengan file sumber.
area = current_h * current_w
aspect_h = int(round((area / source_ratio) ** 0.5))
aspect_w = int(round(aspect_h * source_ratio))
print(f'File asli: H×W={source_h}×{source_w} (W/H={source_ratio:.3f})')
print(f'Target training saat ini: H×W={current_h}×{current_w} (W/H={current_ratio:.3f})')
print(f'Usulan target berasio terjaga: H×W={aspect_h}×{aspect_w} (W/H={aspect_w/aspect_h:.3f})')
print(f'Skala resize saat ini: vertikal={current_h/source_h:.3f}×, horizontal={current_w/source_w:.3f}×')

rgb_aspect = np.asarray(Image.open(row['image']).convert('RGB').resize((aspect_w, aspect_h), Image.Resampling.BILINEAR))
mask_aspect = _resize_array(alpha_native.astype(np.uint8)[..., None] * 255, (aspect_h, aspect_w), True)[..., 0] > 0
depth_aspect_raw = _resize_array(raw_depth, (aspect_h, aspect_w))[..., 0]
depth_aspect_m = depth_aspect_raw * float(row.get('depth_scale_to_m', 0.01))
normal_aspect = _resize_array(raw_normal, (aspect_h, aspect_w))
normal_aspect = F.normalize(torch.from_numpy(normal_aspect).permute(2, 0, 1), dim=0, eps=1e-6).permute(1, 2, 0).numpy()
valid_aspect_depth = (np.isfinite(depth_aspect_raw) & (depth_aspect_raw > 0) & (depth_aspect_raw < 1000)
                     & ~np.isclose(depth_aspect_raw, 65504.0, rtol=0.0, atol=1.0))
valid_aspect = mask_aspect & valid_aspect_depth & np.isfinite(normal_aspect).all(axis=-1)
normal_aspect_rgb = np.clip(normal_aspect, -1, 1) * 0.5 + 0.5
normal_aspect_rgb[~valid_aspect] = 0
depth_aspect_show = np.ma.masked_where(~valid_aspect, depth_aspect_m)
fig, axes = plt.subplots(1, 5, figsize=(20, 6))
axes[0].imshow(rgb_native); axes[0].set_title(f'RGB asli\n{source_h}×{source_w}')
axes[1].imshow(rgb_resized); axes[1].contour(alpha_resized, levels=[0.5], colors='lime', linewidths=.7); axes[1].set_title(f'Resize training\n{current_h}×{current_w}')
axes[2].imshow(rgb_aspect); axes[2].contour(mask_aspect, levels=[0.5], colors='lime', linewidths=.7); axes[2].set_title(f'Rasio terjaga\n{aspect_h}×{aspect_w}')
if valid_aspect.any():
    vlo, vhi = np.percentile(depth_aspect_m[valid_aspect], [2, 98])
else:
    vlo, vhi = 0, 1
im = axes[3].imshow(depth_aspect_show, cmap='viridis', vmin=vlo, vmax=vhi); axes[3].set_title('Depth GT (m)\nrasio terjaga')
fig.colorbar(im, ax=axes[3], fraction=.046, pad=.04)
axes[4].imshow(normal_aspect_rgb); axes[4].set_title('Normal GT\nrasio terjaga')
for ax in axes: ax.axis('off')
fig.suptitle('Audit rasio aspek — belum menjalankan prediksi model'); fig.tight_layout(); plt.show()
if valid_aspect.any():
    print('Aspect-preserved valid GT depth p01/p50/p99 (m):', np.percentile(depth_aspect_m[valid_aspect], [1, 50, 99]).round(3).tolist())
print('Catatan: untuk uji prediksi yang sah, training dan inference harus memakai transformasi rasio-aspek yang sama.')

## Uji checkpoint pada rasio landscape vs portrait
Menjalankan checkpoint fine-tune yang sedang dimuat pada lima sampel yang sama: ukuran training saat ini dan ukuran native portrait. Metrik dihitung terhadap GT yang ditransformasi dengan ukuran masing-masing. Ini menguji sensitivitas checkpoint terhadap rasio input; karena model dilatih pada ukuran landscape, hasil portrait adalah uji di luar preprocessing training, bukan klaim performa setelah training ulang. Jalankan setelah cell Load checkpoint memakai fine-tune.


In [ ]:
# Uji inferensi terkontrol: checkpoint sama, sampel sama, dua ukuran/aspect ratio.
import pandas as pd

LANDSCAPE_SIZE = tuple(int(v) for v in exp['input_size'])  # (H, W), mis. (384, 512)
with Image.open(selected[0]['image']) as im:
    PORTRAIT_SIZE = (im.height, im.width)  # ukuran native file, mis. (512, 384)
for row in selected[1:]:
    with Image.open(row['image']) as im:
        if (im.height, im.width) != PORTRAIT_SIZE:
            raise ValueError('Ukuran native sampel berbeda; jalankan perbandingan per ukuran sumber.')

sizes = {'training_landscape': LANDSCAPE_SIZE, 'native_portrait': PORTRAIT_SIZE}
outputs_by_mode = {}
metric_rows_aspect = []
model.eval()
with torch.inference_mode():
    for mode_name, size_hw in sizes.items():
        ds = RawSynthHumanDataset(selected, input_size=size_hw, augment=False)
        mode_outputs = []
        for i, row in enumerate(selected):
            x, depth_gt, normal_gt, mask = ds[i]
            depth_pred, normal_pred = model(x.unsqueeze(0).to(DEVICE))
            m = mask[0].bool()
            gt = depth_gt[0].float()
            dp = depth_pred[0, 0].detach().cpu().float()
            ng = F.normalize(normal_gt.float(), dim=0, eps=1e-6)
            npred = F.normalize(normal_pred[0].detach().cpu().float(), dim=0, eps=1e-6)
            angle = torch.rad2deg(torch.acos((ng * npred).sum(0).clamp(-1, 1)))
            de = dp[m] - gt[m]
            metric_rows_aspect.append({
                'sample_id': row['subject_id'], 'input_mode': mode_name,
                'input_HxW': f'{size_hw[0]}x{size_hw[1]}',
                'depth_MAE_cm': float(de.abs().mean() * 100),
                'depth_RMSE_cm': float(de.square().mean().sqrt() * 100),
                'normal_mean_deg': float(angle[m].mean()),
                'normal_median_deg': float(angle[m].median()),
                'normal_within_11_25_pct': float((angle[m] < 11.25).float().mean() * 100),
            })
            mode_outputs.append({
                'row': row, 'x': x.cpu(), 'depth_gt': gt.cpu(), 'normal_gt': ng.cpu(),
                'mask': m.cpu(), 'depth_pred': dp, 'normal_pred': npred.cpu(),
            })
        outputs_by_mode[mode_name] = mode_outputs

aspect_table = pd.DataFrame(metric_rows_aspect)
print('Per-sample metrics; nilai lebih kecil lebih baik, kecuali within-11.25° lebih besar lebih baik:')
display(aspect_table.round(3))
print('Rata-rata per-image per input mode:')
display(aspect_table.groupby('input_mode').mean(numeric_only=True).round(3))

# Depth: satu skala warna bersama untuk kedua bentuk input pada tiap sampel.
fig, axes = plt.subplots(len(selected), 6, figsize=(22, 4 * len(selected)), squeeze=False)
for j, title in enumerate(['RGB landscape', 'GT depth landscape', 'Pred depth landscape',
                           'RGB portrait', 'GT depth portrait', 'Pred depth portrait']):
    axes[0, j].set_title(title)
for i in range(len(selected)):
    a = outputs_by_mode['training_landscape'][i]
    b = outputs_by_mode['native_portrait'][i]
    valid_depths = np.concatenate([a['depth_gt'].numpy()[a['mask'].numpy()],
                                   a['depth_pred'].numpy()[a['mask'].numpy()],
                                   b['depth_gt'].numpy()[b['mask'].numpy()],
                                   b['depth_pred'].numpy()[b['mask'].numpy()]])
    vlo, vhi = np.percentile(valid_depths[np.isfinite(valid_depths)], [2, 98])
    for j, (item, key) in enumerate([(a, 'rgb'), (a, 'gt'), (a, 'pred'),
                                     (b, 'rgb'), (b, 'gt'), (b, 'pred')]):
        ax = axes[i, j]
        if key == 'rgb':
            rgb = item['x'].permute(1, 2, 0).numpy() * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
            ax.imshow(np.clip(rgb, 0, 1))
        else:
            arr = item['depth_gt'].numpy() if key == 'gt' else item['depth_pred'].numpy()
            ax.imshow(np.ma.masked_where(~item['mask'].numpy(), arr), cmap='viridis', vmin=vlo, vmax=vhi)
        ax.axis('off')
    axes[i, 0].set_ylabel(selected[i]['subject_id']); axes[i, 0].yaxis.set_visible(True); axes[i, 0].tick_params(left=False, labelleft=False)
fig.suptitle('Depth: checkpoint sama pada resize training vs rasio portrait native; skala warna sama per sampel')
fig.tight_layout(); plt.show()

# Normals: gunakan encoding XYZ→RGB yang konsisten untuk GT dan prediksi.
fig, axes = plt.subplots(len(selected), 6, figsize=(22, 4 * len(selected)), squeeze=False)
for j, title in enumerate(['RGB landscape', 'GT normal landscape', 'Pred normal landscape',
                           'RGB portrait', 'GT normal portrait', 'Pred normal portrait']):
    axes[0, j].set_title(title)
for i in range(len(selected)):
    a = outputs_by_mode['training_landscape'][i]
    b = outputs_by_mode['native_portrait'][i]
    for j, item in enumerate([a, b]):
        rgb = item['x'].permute(1, 2, 0).numpy() * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
        ng = np.moveaxis(np.clip(item['normal_gt'].numpy(), -1, 1) * 0.5 + 0.5, 0, -1)
        npred = np.moveaxis(np.clip(item['normal_pred'].numpy(), -1, 1) * 0.5 + 0.5, 0, -1)
        mask_np = item['mask'].numpy()
        ng[~mask_np] = 0; npred[~mask_np] = 0
        for col, panel in zip((j * 3, j * 3 + 1, j * 3 + 2), (np.clip(rgb, 0, 1), ng, npred)):
            axes[i, col].imshow(panel); axes[i, col].axis('off')
    axes[i, 0].set_ylabel(selected[i]['subject_id']); axes[i, 0].yaxis.set_visible(True); axes[i, 0].tick_params(left=False, labelleft=False)
fig.suptitle('Normals: encoding warna XYZ sama; bandingkan pola GT dan prediksi')
fig.tight_layout(); plt.show()


## Lokalisasi error: batas, bagian dalam, dan area vertikal
Cell ini membedakan error depth/normal di dekat siluet dari error di bagian dalam mask. Ia juga membuat pembagian atas/tengah/bawah berdasarkan bounding box mask—**ini hanya zona kasar, bukan label kepala/torso/kaki**—dan mengukur kesesuaian gradien depth. Jalankan setelah cell visualisasi lima sample.


In [ ]:
# Diagnostik error per sample dan per wilayah; tidak mengubah model/checkpoint.
import pandas as pd

rows_diag = []
fig, axes = plt.subplots(len(preds), 4, figsize=(16, 4 * len(preds)), squeeze=False)
for i, (row, out) in enumerate(zip(selected, preds)):
    m = torch.as_tensor(out['mask'][0]).bool()
    mt = m[None, None].float()
    # Piksel mask yang bertetangga dengan background dianggap boundary.
    eroded = (1.0 - F.max_pool2d(1.0 - mt, kernel_size=3, stride=1, padding=1))[0, 0] > 0.5
    boundary = m & ~eroded
    interior = m & eroded
    gt = torch.as_tensor(out['depth_gt'][0]).float()
    pdp = torch.as_tensor(out['depth_pred'][0]).float()
    ng = F.normalize(torch.as_tensor(out['normal_gt']).float(), dim=0, eps=1e-6)
    npred = F.normalize(torch.as_tensor(out['normal_pred']).float(), dim=0, eps=1e-6)
    angle = torch.rad2deg(torch.acos((ng * npred).sum(0).clamp(-1, 1)))
    err = (pdp - gt).abs()
    valid_np = m.numpy(); gt_np = gt.numpy(); pred_np = pdp.numpy()
    yy, xx = np.where(valid_np)
    y0, y1 = int(yy.min()), int(yy.max()) + 1
    pred_median = float(torch.median(pdp[m]))
    gt_median = float(torch.median(gt[m]))
    ratio = pred_median / max(gt_median, 1e-8)
    record = {'sample_id': row['subject_id'], 'GT_median_m': gt_median, 'pred_median_m': pred_median, 'pred/GT_median': ratio}
    for label, region in [('boundary', boundary), ('interior', interior)]:
        if region.any():
            record[f'depth_MAE_{label}_cm'] = float(err[region].mean() * 100)
            record[f'normal_mean_{label}_deg'] = float(angle[region].mean())
        else:
            record[f'depth_MAE_{label}_cm'] = float('nan')
            record[f'normal_mean_{label}_deg'] = float('nan')
    for band, (lo, hi) in {'upper_bbox': (0.0, 1/3), 'middle_bbox': (1/3, 2/3), 'lower_bbox': (2/3, 1.01)}.items():
        ylo = y0 + int((y1-y0)*lo); yhi = min(y1, y0 + int((y1-y0)*hi))
        region = m.clone(); region[:ylo] = False; region[yhi:] = False
        record[f'depth_MAE_{band}_cm'] = float(err[region].mean()*100) if region.any() else float('nan')
        record[f'normal_mean_{band}_deg'] = float(angle[region].mean()) if region.any() else float('nan')
    # Gradient agreement checks whether depth transitions occur in similar places.
    gx_gt, gy_gt = np.gradient(gt_np); gx_pr, gy_pr = np.gradient(pred_np)
    mag_gt = np.hypot(gx_gt, gy_gt); mag_pr = np.hypot(gx_pr, gy_pr)
    grad_mask = interior.numpy() & np.isfinite(mag_gt) & np.isfinite(mag_pr)
    if grad_mask.sum() > 10 and np.std(mag_gt[grad_mask]) > 1e-8 and np.std(mag_pr[grad_mask]) > 1e-8:
        record['depth_gradient_corr_interior'] = float(np.corrcoef(mag_gt[grad_mask], mag_pr[grad_mask])[0, 1])
    else:
        record['depth_gradient_corr_interior'] = float('nan')
    rows_diag.append(record)

    rgb = out['x'].permute(1, 2, 0).numpy() * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
    axes[i, 0].imshow(np.clip(rgb, 0, 1)); axes[i, 0].contour(valid_np, levels=[0.5], colors='lime', linewidths=.6); axes[i, 0].set_title(row['subject_id'] + '\nRGB + mask')
    axes[i, 1].imshow(np.ma.masked_where(~valid_np, err.numpy()), cmap='inferno'); axes[i, 1].set_title('|Depth error| (m)')
    a = angle.numpy().copy(); a[~valid_np] = np.nan
    axes[i, 2].imshow(a, cmap='inferno', vmin=0, vmax=90); axes[i, 2].set_title('Normal angular error (°)')
    overlay = np.zeros((*valid_np.shape, 3), dtype=np.float32); overlay[interior.numpy()] = (0.1, 0.3, 1.0); overlay[boundary.numpy()] = (1.0, 0.2, 0.1)
    axes[i, 3].imshow(overlay); axes[i, 3].set_title('Interior (blue) / boundary (red)')
    for ax in axes[i]: ax.axis('off')
fig.suptitle('Lokasi error pada lima sample (warna error dinormalisasi per panel)'); fig.tight_layout(); plt.show()

diagnostic_table = pd.DataFrame(rows_diag)
display(diagnostic_table.round(3))
print('Baca pred/GT median sebagai indikasi bias skala, bukan akurasi.')
print('Gradient correlation mendekati 1 berarti pola perubahan depth lebih mirip; ini bukan metrik resmi.')
print('Boundary vs interior membantu menguji detail siluet, tetapi tidak mengisolasi lipatan di bagian dalam.')


## Uji sensitivitas resolusi inference
Cell opsional ini menjalankan sample yang sama pada resolusi setengah, resolusi training, dan 1,5× resolusi training. Prediksi dikembalikan ke ukuran GT sebelum dihitung metrik. Ini menguji sensitivitas resolusi input; **bukan** bukti bahwa resolusi lebih tinggi akan memperbaiki hasil setelah training.


In [ ]:
# Sensitivitas resolusi pada held-out SynthHuman; GPU/Colab diperlukan untuk ukuran terbesar.
base_h, base_w = (int(v) for v in exp['input_size'])
RESOLUTION_FACTORS = (0.5, 1.0, 1.5)
resolution_rows = []
resolution_maps = {}
model.eval()
for i, (row, out) in enumerate(zip(selected, preds)):
    if i >= min(5, len(selected)):
        break
    x0 = out['x'].unsqueeze(0).to(DEVICE)
    dgt = torch.as_tensor(out['depth_gt'][0]).float().to(DEVICE)
    ngt = F.normalize(torch.as_tensor(out['normal_gt']).float().to(DEVICE), dim=0, eps=1e-6).to(DEVICE)
    mask = torch.as_tensor(out['mask'][0]).bool().to(DEVICE)
    for factor in RESOLUTION_FACTORS:
        h = max(32, int(round(base_h * factor / 32)) * 32)
        w = max(32, int(round(base_w * factor / 32)) * 32)
        x_scaled = F.interpolate(x0, size=(h, w), mode='bilinear', align_corners=False)
        with torch.inference_mode():
            dp, npred = model(x_scaled)
            dp = F.interpolate(dp, size=dgt.shape, mode='bilinear', align_corners=False)[0, 0]
            npred = F.interpolate(npred, size=dgt.shape, mode='bilinear', align_corners=False)[0]
            npred = F.normalize(npred, dim=0, eps=1e-6)
        de = (dp[mask] - dgt[mask])
        dot = (npred * ngt).sum(0).clamp(-1, 1)
        ne = torch.rad2deg(torch.acos(dot[mask]))
        resolution_rows.append({'sample_id': row['subject_id'], 'input_HxW': f'{h}x{w}', 'factor_vs_train': factor,
                                'depth_MAE_cm': float(de.abs().mean()*100), 'depth_RMSE_cm': float(de.square().mean().sqrt()*100),
                                'normal_mean_deg': float(ne.mean()), 'normal_within_11_25_pct': float((ne < 11.25).float().mean()*100)})
        if i == 0:
            resolution_maps[(h, w)] = dp.detach().cpu().numpy()

display(pd.DataFrame(resolution_rows).round(3))
# Visual comparison for first sample, shared GT-derived color range.
first = preds[0]; first_mask = first['mask'][0].bool().numpy(); first_gt = first['depth_gt'][0].numpy()
lo, hi = np.percentile(first_gt[first_mask], [2, 98])
fig, axes = plt.subplots(1, len(resolution_maps)+1, figsize=(5*(len(resolution_maps)+1), 5), squeeze=False)
axes[0, 0].imshow(np.ma.masked_where(~first_mask, first_gt), cmap='viridis', vmin=lo, vmax=hi); axes[0, 0].set_title('Depth GT')
for ax, ((h, w), dm) in zip(axes[0, 1:], resolution_maps.items()):
    ax.imshow(np.ma.masked_where(~first_mask, dm), cmap='viridis', vmin=lo, vmax=hi); ax.set_title(f'Prediksi input {h}×{w}')
for ax in axes[0]: ax.axis('off')
fig.suptitle('Perbandingan resolusi pada sample pertama; skala warna sama'); fig.tight_layout(); plt.show()
print('Resolusi lebih besar dapat menambah biaya komputasi. Perbandingan ini hanya diagnosis inference, bukan pengganti eksperimen training terkontrol.')


## Uji satu foto nyata dengan mask orang (tanpa ground truth)

Cell berikut meminta satu foto di-upload, memakai DeepLabV3 pra-latih untuk mendapatkan mask kelas orang, lalu menampilkan prediksi depth dan normal hanya di dalam mask itu. Mask segmentasi ini hanya untuk membatasi visualisasi; RGB penuh tetap diberikan ke model geometri. Ini bukan ground truth dan tidak mengubah prediksi model. Tanpa ground truth kita tidak menghitung MAE/RMSE atau normal error. Prediksi depth dalam meter juga belum tentu terkalibrasi untuk foto nyata.

Secara default memakai checkpoint yang sudah dimuat di atas. Setelah checkpoint fine-tune 384x512 selesai dan disalin ke Drive, ubah `REAL_CHECKPOINT_PATH` ke file tersebut untuk mencoba model baru.

In [ ]:
from google.colab import files
from PIL import Image
import io
from torchvision.models.segmentation import deeplabv3_resnet50, DeepLabV3_ResNet50_Weights

# Default: checkpoint loaded in the previous cells. Change this after
# exporting the fine-tuned best_checkpoint.pt to Google Drive.
REAL_CHECKPOINT_PATH = CHECKPOINT_LOCAL
# Example after export:
# REAL_CHECKPOINT_PATH = DRIVE_PROJECT / 'results/fastvit_t8_fpn_depth_normals_local50000_seed42_finetune_lr2e6_384x512_100ep_v1/best_checkpoint.pt'

real_checkpoint_path = Path(REAL_CHECKPOINT_PATH)
real_state = torch.load(real_checkpoint_path, map_location='cpu', weights_only=False)
real_exp = real_state.get('experiment', exp)
if real_checkpoint_path.resolve() == CHECKPOINT_LOCAL.resolve():
    real_model = model
else:
    real_model = PortraitGeometryModel(real_exp['backbone'], pretrained=False, fpn_width=int(real_exp['fpn_width']), fpn_fusion=real_exp.get('fpn_fusion', 'add'))
    real_model.load_state_dict(real_state['model'], strict=True)
    real_model.to(DEVICE).eval()

uploaded = files.upload()
if not uploaded:
    raise ValueError('Upload satu foto untuk menjalankan visualisasi.')
photo_name, photo_bytes = next(iter(uploaded.items()))
photo = Image.open(io.BytesIO(photo_bytes)).convert('RGB')

# Preserve the photo aspect ratio; pad with ImageNet mean color instead of stretching it.
target_h, target_w = (int(v) for v in real_exp['input_size'])
source_w, source_h = photo.size
scale = min(target_w / source_w, target_h / source_h)
resized_w = max(1, round(source_w * scale))
resized_h = max(1, round(source_h * scale))
photo_resized = photo.resize((resized_w, resized_h), Image.Resampling.LANCZOS)
left = (target_w - resized_w) // 2
top = (target_h - resized_h) // 2
mean_rgb = tuple(round(v * 255) for v in (0.485, 0.456, 0.406))
canvas = Image.new('RGB', (target_w, target_h), mean_rgb)
canvas.paste(photo_resized, (left, top))
rgb = np.asarray(canvas, dtype=np.float32) / 255.0
mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
x = torch.from_numpy(((rgb - mean) / std).copy()).permute(2, 0, 1).unsqueeze(0).to(DEVICE)

real_model.eval()
with torch.inference_mode():
    depth_pred, normal_pred = real_model(x)
depth_map = depth_pred[0, 0].float().cpu().numpy()[top:top + resized_h, left:left + resized_w]
normal_map = normal_pred[0].float().cpu().numpy()[:, top:top + resized_h, left:left + resized_w]

# Use a pretrained person segmenter only to create a display mask.
# It is separate from the trained depth/normal model and its result is not ground truth.
seg_weights = DeepLabV3_ResNet50_Weights.DEFAULT
person_segmenter = deeplabv3_resnet50(weights=seg_weights).to(DEVICE).eval()
person_idx = seg_weights.meta['categories'].index('person')
seg_input = seg_weights.transforms()(photo_resized).unsqueeze(0).to(DEVICE)
with torch.inference_mode():
    logits = person_segmenter(seg_input)['out']
    logits = F.interpolate(logits, size=(resized_h, resized_w), mode='bilinear', align_corners=False)
    person_prob = logits.softmax(dim=1)[0, person_idx].cpu().numpy()
PERSON_THRESHOLD = 0.5
person_mask = person_prob >= PERSON_THRESHOLD
if person_mask.sum() == 0:
    raise RuntimeError('Segmenter tidak menemukan orang. Coba foto yang lebih terang/jelas atau turunkan PERSON_THRESHOLD, misalnya ke 0.35.')
normal_rgb = np.moveaxis(np.clip(normal_map, -1, 1) * 0.5 + 0.5, 0, -1)
normal_rgb[~person_mask] = 0
masked_depth = np.ma.masked_where(~person_mask | ~np.isfinite(depth_map), depth_map)
depth_lo, depth_hi = np.percentile(depth_map[person_mask & np.isfinite(depth_map)], [2, 98])

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
axes[0].imshow(photo_resized)
axes[0].imshow(np.ma.masked_where(~person_mask, person_mask), cmap='spring', alpha=0.35, vmin=0, vmax=1)
axes[0].set_title('Foto + overlay mask orang')
axes[1].imshow(person_mask, cmap='gray', vmin=0, vmax=1)
axes[1].set_title(f'Mask segmentasi (threshold={PERSON_THRESHOLD})')
depth_artist = axes[2].imshow(masked_depth, cmap='viridis', vmin=depth_lo, vmax=depth_hi)
axes[2].set_title('Depth pada orang (m; belum terverifikasi)')
fig.colorbar(depth_artist, ax=axes[2], fraction=0.046, pad=0.04, label='Predicted depth (m)')
axes[3].imshow(normal_rgb)
axes[3].set_title('Normal pada orang (XYZ → RGB)')
for ax in axes:
    ax.axis('off')
fig.suptitle(f'Real-photo visual check: {photo_name} | checkpoint: {real_checkpoint_path.name}')
fig.tight_layout()
plt.show()
print('Mask berasal dari model segmentasi pra-latih, bukan label ground truth. Visual-only: tidak ada metrik akurasi tanpa ground truth.')
print(f'Displayed predicted depth range (2nd-98th percentile): {depth_lo:.3f} to {depth_hi:.3f} m; real-world scale is not verified.')